# Comparative Study of Four ML Algorithms for IMDb Sentiment Classification

**Classes:** Positive and Negative

This notebook performs the experiment and computes actual results from the dataset you upload. No metrics are pre-filled.

In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib seaborn

## 1. Upload the IMDb CSV

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload IMDB Dataset.csv

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

In [ ]:
df = pd.read_csv("IMDB Dataset.csv")
df = df[["review", "sentiment"]].dropna().copy()
df["sentiment"] = df["sentiment"].astype(str).str.lower().str.strip()
df = df[df["sentiment"].isin(["positive", "negative"])]

# Balanced reproducible sample: 5,000 per class
df = (df.groupby("sentiment", group_keys=False)
        .sample(n=5000, random_state=42)
        .sample(frac=1, random_state=42)
        .reset_index(drop=True))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["review"] = df["review"].apply(clean_text)
print(df.shape)
print(df["sentiment"].value_counts())
df.head()

In [ ]:
# Split BEFORE fitting TF-IDF to avoid test-data leakage
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["review"], df["sentiment"],
    test_size=0.20, random_state=42, stratify=df["sentiment"]
)

tfidf = TfidfVectorizer(
    max_features=10000, stop_words="english", ngram_range=(1, 2)
)
X_train = tfidf.fit_transform(X_train_text)
X_test = tfidf.transform(X_test_text)

print("Train:", X_train.shape, "Test:", X_test.shape)

In [ ]:
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5, metric="cosine", algorithm="brute"),
    "SVM": LinearSVC(C=1.0, random_state=42),
    "Naive Bayes": MultinomialNB()
}

results = []
predictions = {}
for name, model in models.items():
    print("Training", name)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    predictions[name] = pred
    results.append({
        "Algorithm": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, pos_label="positive", zero_division=0),
        "Recall": recall_score(y_test, pred, pos_label="positive", zero_division=0),
        "F1-score": f1_score(y_test, pred, pos_label="positive", zero_division=0)
    })

results_df = pd.DataFrame(results)
display(results_df.round(4))

In [ ]:
ax = results_df.set_index("Algorithm")[["Accuracy", "Precision", "Recall", "F1-score"]].plot(
    kind="bar", figsize=(11, 6), ylim=(0, 1), width=0.8
)
ax.set_title("Performance Comparison of Four Text Classifiers")
ax.set_ylabel("Score")
ax.set_xlabel("Algorithm")
ax.tick_params(axis="x", rotation=0)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, (name, pred) in zip(axes.flatten(), predictions.items()):
    cm = confusion_matrix(y_test, pred, labels=["negative", "positive"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Negative", "Positive"],
                yticklabels=["Negative", "Positive"])
    ax.set_title(name)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("Actual label")
plt.tight_layout()
plt.show()

## 8. What-if experiment: change SVM C from 1.0 to 0.1

In [ ]:
what_if_rows = []
for c_value in [1.0, 0.1]:
    model = LinearSVC(C=c_value, random_state=42)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    what_if_rows.append({
        "SVM setting": f"C={c_value}",
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, pos_label="positive", zero_division=0),
        "Recall": recall_score(y_test, pred, pos_label="positive", zero_division=0),
        "F1-score": f1_score(y_test, pred, pos_label="positive", zero_division=0)
    })
what_if_df = pd.DataFrame(what_if_rows)
display(what_if_df.round(4))

## 9. Save outputs

In [ ]:
from pathlib import Path
out = Path("results")
out.mkdir(exist_ok=True)
results_df.to_csv(out / "algorithm_comparison.csv", index=False)
what_if_df.to_csv(out / "svm_what_if_comparison.csv", index=False)
print("Saved CSV result files in ./results")